# STS Silent GPU on Colab: build + speed test

先选 GPU：菜单 **代码执行程序 → 更改运行时类型 → GPU**，然后 **全部运行**。

只做编译和模拟器测速，不训练、不连服务器。最后一格的数字发给 Claude。

参考：服务器单张 V100，2048 个并行局约 23.6 万决策/秒（2026-10-18 旧版本，仅供量级对比）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!nproc; free -g | head -2
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y lsb-release wget software-properties-common gnupg time >/dev/null
wget -qO /tmp/llvm.sh https://apt.llvm.org/llvm.sh
bash /tmp/llvm.sh 21 >/dev/null
pip -q install 'cmake>=3.31' pybind11
/usr/lib/llvm-21/bin/clang++ --version | head -1; cmake --version | head -1; nvcc --version | tail -1

In [ ]:
%%bash
set -e
cd /content
rm -rf sts-silent-gpu
git clone -q --depth 1 -b codex/e4 https://github.com/destire-mio/sts-silent-gpu
cd sts-silent-gpu && git log --oneline -1

In [ ]:
%%bash
set -e
cd /content/sts-silent-gpu
export PATH=/usr/lib/llvm-21/bin:$PATH OMP_NUM_THREADS=1
arch=$(python -c 'import torch; a,b=torch.cuda.get_device_capability(); print(a*10+b)')
cuda=$(dirname $(dirname $(realpath $(command -v nvcc))))
py=$(command -v python)
echo arch=$arch cuda=$cuda
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_COMPILER=clang++ \
  -DCMAKE_CUDA_COMPILER=clang++ -DCUDAToolkit_ROOT=$cuda \
  -DCMAKE_CUDA_COMPILER_LAUNCHER="$py;$PWD/tools/cuda_compile.py;compile" \
  -DCMAKE_CUDA_ARCHITECTURES=$arch -DPython_EXECUTABLE=$py -DBUILD_TESTING=OFF > build.log 2>&1 || { tail -40 build.log; exit 1; }
/usr/bin/time -f 'build wall %E, peak RSS %M KiB' cmake --build build --target spire spire_gpu --parallel 2 >> build.log 2>&1 || { tail -60 build.log; exit 1; }
tail -3 build.log; ls -la build/*.so

In [ ]:
%%bash
cd /content/sts-silent-gpu
PYTHONPATH=build:. python tools/benchmark_cuda.py --sizes 2048 8192 32768 --seconds 10